# MODFLOW 6 Lab:

This workshop explores how to extend the capabilities of MODFLOW 6 for advanced groundwater, solute transport, and geothermal modeling. Participants will learn how to use Python and the pymf6 library to implement custom boundary conditions, such as direction-dependent riverbed conductance, and couple MODFLOW with external models or analytical solutions. The focus is on practical, flexible tools to solve complex subsurface problems with minimal code and maximum adaptability.


## Introduction round

*Speakers*: 

Dr.-Ing. Mike Müller (hydrocomputing): developer of pymf6

Lúcia Pedrosa (TUBAF): developer of pymf6-based model features

# Program

## Thursday
9:30 – 9:45 Welcome & Introduction

9:45 -10:30 Technical set-up

10:30 -10:45 Break

10:45 – 12:00 pymf6 Principles:
 + Running MODFLOW 6 via pymf6 within a Jupyter Notebook
 + Basics for working with pymf6
 + Inspecting internal MODFLOW values during model run time

12:00 – 13:00 Lunch

13:00 – 15:00 Scenario flow: Controlling the well withdraw based on a target groundwater level

15:00 – 15:30 Break

15:30 – 17:00 Scenario flow: Dynamic river bed conductance for gaining and loosing river sections

## Friday 

9:30 – 10:00 Recap & Discussion

10:00 – 11:00 Scenario transport: Optimize the pumping rate of hydraulic barrier for containing a contamination

11:00 – 11:15 Break 

11.15 – 13:00 Scenario geothermal: Extend MODFLOW with a thermal Cauchy boundary condition

13:00 – 14:00 Lunch 

14:00 – 15:30 Outlook: 

 + Coupling analytic solutions to reduce model size
 + Reactive transport modeling with pymf6

15:30 – 15:45 Break 

15:45 – 16:30 Q&A: Ask questions about your model customization needs and get answers from author of pymf6

## Workshop Goals 

1. Understand and extend the modular design of MODFLOW 6.

2. Learn how to build and couple groundwater, solute transport, and geothermal flow models.

3. Use Python + PyMF6 to automate modeling workflows and customize boundary conditions.

4. Implement direction-dependent and dynamic boundary conditions programmatically.

5. Couple MODFLOW 6 with external analytical or numerical models.

6. Apply efficient coding patterns to build adaptable, reproducible hydrogeologic simulations.

## Introduction to Modflow 6

## What is MODFLOW 6?

MODFLOW 6 (MF6) is the latest generation of the U.S. Geological Survey (USGS) MODFLOW groundwater modeling framework. It represents a complete redesign of the classic MODFLOW series (MODFLOW-2005, MODFLOW-NWT, MODFLOW-USG, etc.), combining the best features of previous versions into a unified, flexible, and modular system.

## Key characteristics:

1. Unified Framework:

MODFLOW 6 provides a common platform for simulating multiple hydrologic processes such as groundwater flow (GWF), groundwater transport (GWT), and exchanges between models.

2. Modular and Object-Oriented Architecture:
Each process, model, and package is implemented as a modular unit, allowing flexible model composition. You can run multiple models (e.g., GWF + GWT) within the same simulation and define exchanges between them (e.g., flow coupling, solute transport).

3. Numerical Improvements:

 + Enhanced iterative solver (IMS) package with better control over convergence and performance.

 + Advanced time-stepping and adaptive convergence checks.

 + Support for structured (DIS), vertex-based (DISV), and unstructured (DISU) grids.

4. Inter-model Coupling:

 + Native support for GWF–GWF exchanges (linking aquifers or model domains).

 + Support for GWF–GWT coupling (flow and transport interactions).

 + Facilitates integrated groundwater–surface water or multi-layered aquifer modeling.

5. Enhanced Output Control and Post-processing:
   
Output control (OC) allows users to customize which variables are written and at what frequency. Head, budget, and concentration outputs can be easily parsed using FloPy or PyMF6.

## Internal Structure

MODFLOW 6 uses three primary components:

1. Simulation (SIM): The top-level container that manages time, models, and exchanges.

2. Model (GWF, GWT, etc.): Represents a specific process (flow, transport, etc.). Each model is composed of packages.

3. Package: Defines parameters, boundary conditions, and solver settings.

## Modular structure 

High-level hierarchy:

+ Simulation (controls time, models, exchanges)

   + Model(s) (GWF, GWT)

     + Package(s) (DIS, NPF, STO, WEL, CHD, etc.)

Packages are the building blocks where each represents a physical process or numerical control (e.g., discretization, boundary conditions, solver settings).

## Package structure 

Common package categories:

+ Discretization: DIS, DISV, DISU

+ Properties: NPF (flow properties), STO (storage)

+ Initial & Boundary Conditions: IC, CHD, WEL, RCH, DRN, RIV, GHB, EVT

+ Solver & Output: IMS, OC

+ Coupling & Exchanges: EXG, GWF-GWT connectors

## FloPy: Installation and Quick Start

💻 To install FloPy, use pip directly from your Python environment:

Verify the installation:

In [8]:
import flopy 
print('FloPy version:', flopy.__version__)

FloPy version: 3.9.3


In [4]:
import flopy
import numpy as np


# Create a simulation and model
tsim = flopy.mf6.MFSimulation(sim_name='tutorial_sim', exe_name='mf6', version='mf6', sim_ws='model_ws')
gwf = flopy.mf6.ModflowGwf(tsim, modelname='tutorial_model', save_flows=True)


# Add Time Discretization Package (TDIS)
flopy.mf6.ModflowTdis(tsim, time_units='DAYS', nper=1, perioddata=[(1.0, 1, 1.0)])


# Add the IMS Solver
flopy.mf6.ModflowIms(tsim)


# Define the model grid
dis = flopy.mf6.ModflowGwfdis(gwf, nlay=1, nrow=5, ncol=5, delr=100.0, delc=100.0, top=10.0, botm=0.0)


# Initial Conditions
ic = flopy.mf6.ModflowGwfic(gwf, strt=10.0)


# Hydraulic Properties
npf = flopy.mf6.ModflowGwfnpf(gwf, icelltype=1, k=10.0)


# Constant Head Boundary (CHD)
chd_spd = [((0, i, 0), 10.0) for i in range(5)] + [((0, i, 4), 5.0) for i in range(5)]
flopy.mf6.ModflowGwfchd(gwf, stress_period_data=chd_spd)


# Output Control
flopy.mf6.ModflowGwfoc(gwf, head_filerecord='head.hds', saverecord=[('HEAD', 'ALL')])


# Write and run the model
tsim.write_simulation()
tsim.run_simulation()


print('Simulation complete. Output stored in model_ws/')

FloPy skeleton created — inspect objects:
sim_name = mf6_test
sim_path = C:\Users\lucialabarca\re-run noteboks\pymf6-validation\src\notebooks
exe_name = mf6

###################
Package mfsim.nam
###################

package_name = mfsim.nam
filename = mfsim.nam
package_type = nam
model_or_simulation_package = simulation
simulation_name = mf6_test


###################
Package mf6_test.tdis
###################

package_name = mf6_test.tdis
filename = mf6_test.tdis
package_type = tdis
model_or_simulation_package = simulation
simulation_name = mf6_test


@@@@@@@@@@@@@@@@@@@@
Model gwf_1
@@@@@@@@@@@@@@@@@@@@

name = gwf_1
model_type = gwf6
version = mf6
model_relative_path = .

###################
Package dis
###################

package_name = dis
filename = gwf_1.dis
package_type = dis
model_or_simulation_package = model
model_name = gwf_1


###################
Package ic
###################

package_name = ic
filename = gwf_1.ic
package_type = ic
model_or_simulation_package = model
mod

## Intro to programming 